In [ ]:
import pandas as pd

df = pd.read_csv("../data/SeoulBikeData.csv", encoding="latin1")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# SELECT
#     [Functioning Day],
#     COUNT(*) AS [count],
#     AVG([Rented Bike Count]) AS [mean],
#     MIN([Rented Bike Count]) AS [min],
#     MAX([Rented Bike Count]) AS [max]
# FROM SeoulBikeData
# GROUP BY [Functioning Day];

df.groupby("Functioning Day")["Rented Bike Count"].agg(
    ["count", "mean", "min", "max"]
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

ax = df[df["Functioning Day"] == "Yes"].groupby("Hour")["Rented Bike Count"].mean().plot(
    kind="line",
    marker="o",
    title="Average bike rentals by hour",
    xlabel="Hour",
    ylabel="Average rented bikes",
    grid=True
)

ax.set_xticks(range(24))

plt.show()

## Initial Data Analysis (EDA)

We started by exploring the Seoul Bike Sharing dataset to understand what data we have and what might affect bike rentals.

### Dataset overview
- 8,760 rows (hourly data for one year).
- 14 columns.
- No missing values.
- Our target is **Rented Bike Count**.

### What we found

**Bike rentals:**
- Average: about 705 bikes per hour, Median: about 505 bikes, Minimum: 0, Maximum: 3,556.

**Functioning days:**
- There are 295 hours when the rental service was not operating (no bikes were rented).
- When the service was operating, at least 2 bikes were rented per hour.

**Time of day:**
- The lowest demand is around 4:00–5:00.
- There is a morning peak around 8:00.
- The highest peak is around 18:00.
- This might be because people use bikes to travel to and from work.

### Initial conclusions

- **Hour** seems to be an important feature for predicting bike rentals.
- The relationship between time and rental demand is not linear.
- We should also investigate how temperature, weather and seasons affect demand.

In [ ]:
# Bike Rentals vs Temperature

working_days = df[df["Functioning Day"] == "Yes"]

plt.figure(figsize=(14, 5))

plt.scatter(
    working_days["Temperature(°C)"],
    working_days["Rented Bike Count"],
    alpha=0.25,
    s=2
)

plt.title("Bike Rentals vs Temperature")
plt.xlabel("Temperature (°C)")
plt.ylabel("Rented Bike Count")
plt.grid(True)
plt.show()

Temperature seems to affect **the maximum number of bike rentals** rather than the actual demand.  
Low rental counts happen at almost any temperature, but high rental counts are mostly observed in warmer weather.  
The upper limit increases up to around 25°C and then appears to decrease.  
Temperature also depends on the time of day, so the relationship between temperature and bike rentals might be partly explained by the hour.

In [ ]:
# Bike Rentals vs Temperature by Hour

plt.figure(figsize=(14, 5))

colors = {
    8: "blue",
    14: "green",
    18: "red"
}

for hour in [8, 14, 18]:
    data = working_days[working_days["Hour"] == hour]

    plt.scatter(
        data["Temperature(°C)"],
        data["Rented Bike Count"],
        color=colors[hour],
        alpha=0.8,
        s=10,
        label=f"{hour}:00"
    )

plt.title("Bike Rentals vs Temperature by Hour")
plt.xlabel("Temperature (°C)")
plt.ylabel("Rented Bike Count")
plt.legend()
plt.grid(True)
plt.show()

The graph shows that the relationship between temperature and bike rentals follows roughly the same pattern at all three selected hours.

In [ ]:
# Weekdays vs Weekends

df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

working_days = df[df["Functioning Day"] == "Yes"].copy()

working_days["Day Type"] = (working_days["Date"].dt.day_of_week >= 5).map({
    False: "Weekday",
    True: "Weekend"
})

ax = working_days.groupby(["Hour", "Day Type"])["Rented Bike Count"].mean().unstack().plot(
    figsize=(14, 5),
    marker="o",
    title="Bike Rentals: Weekdays vs Weekends",
    xlabel="Hour",
    ylabel="Average rented bikes",
    grid=True
)

ax.set_xticks(range(24))
plt.show()

## Weekdays vs Weekends

The graph shows a clear difference between weekdays and weekends. On weekdays, there are two peaks at 8:00 and 18:00, probably(?) because people travel to and from work. On weekends, there are no sharp peaks, and bike rentals increase gradually during the day.

Apart from these peaks, the graphs for weekdays and weekends follow roughly the same pattern.

In [ ]:
# Rain vs No Rain

from IPython.display import display, Markdown

working_days["Rain"] = working_days["Rainfall(mm)"].apply(
    lambda rain: "Rain" if rain > 0 else "No Rain"
)

display(Markdown("### Total observations"))
display(working_days["Rain"].value_counts())

display(Markdown("### Observations by hour"))
display(working_days.groupby(["Hour", "Rain"]).size().unstack())

ax = working_days.groupby(["Hour", "Rain"])["Rented Bike Count"].mean().unstack().plot(
    figsize=(14, 5),
    marker="o",
    title="Bike Rentals: Rain vs No Rain",
    xlabel="Hour",
    ylabel="Average rented bikes",
    grid=True
)
ax.set_xticks(range(24))
plt.show()

## Rain vs No Rain

The graph shows that rain has a strong effect on bike rentals. The number of rentals is much lower when it rains, but the overall pattern during the day is quite similar.